<h3 style="color:#6FA8DC; font-weight:bold">02_Missing_Data_Mean_Median_Imputation</h3>

Handling Missing Data → Univariate Numerical Imputation

Reference basis: the provided CampusX numerical-imputation notebooks and `titanic_toy.csv`.

<h5 style="color:#78B89A; font-weight:bold;">1. What is univariate imputation? → one feature at a time</h5>

Univariate imputation fills missing values in a feature using information from **that same feature**.

Example:

`Age = [22, 38, NaN, 35, 28]`

The missing `Age` is filled using a rule based on the observed `Age` values.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

In [ ]:
df = pd.read_csv('titanic_toy.csv')
df.head()

In [ ]:
df.isnull().mean() * 100

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

X_train.isnull().mean() * 100

<h5 style="color:#78B89A; font-weight:bold;">2. Why impute after train/test split? → avoid leakage</h5>

The value used to fill missing data must be **learned from the training set only**.

```text
X_train → learn imputation value
X_train → transform
X_test  → use the SAME learned value
```

Never calculate a mean/median using the complete dataset before splitting. That allows information from the test set to influence training.

<h5 style="color:#78B89A; font-weight:bold;">3. Method: Mean / Median Imputation</h5>

<h5 style="color:#78B89A; font-weight:bold;">4. Mean imputation → replace NaN with average</h5>

Formula:

`mean = sum(observed values) / number of observed values`

Example: `[10, 20, NaN, 30]` → mean = 20 → `[10, 20, 20, 30]`

In [ ]:
mean_age = X_train['Age'].mean()
mean_fare = X_train['Fare'].mean()

X_train['Age_mean'] = X_train['Age'].fillna(mean_age)
X_train['Fare_mean'] = X_train['Fare'].fillna(mean_fare)

print('Age mean:', mean_age)
print('Fare mean:', mean_fare)

<h5 style="color:#78B89A; font-weight:bold;">5. Median imputation → replace NaN with middle value</h5>

Median is the middle value after sorting the observed values.

Median is often preferred when a numerical feature is skewed or contains outliers because it is less affected by extreme values than the mean.

In [ ]:
median_age = X_train['Age'].median()
median_fare = X_train['Fare'].median()

X_train['Age_median'] = X_train['Age'].fillna(median_age)
X_train['Fare_median'] = X_train['Fare'].fillna(median_fare)

print('Age median:', median_age)
print('Fare median:', median_fare)

<h5 style="color:#78B89A; font-weight:bold;">6. Reference notebook → effect on variance and distribution</h5>

The provided reference compares variance, KDE distributions, covariance/correlation and boxplots. These checks help us see whether imputation changes the feature.

In [ ]:
print('Original Age variance:', X_train['Age'].var())
print('Mean-imputed Age variance:', X_train['Age_mean'].var())
print('Median-imputed Age variance:', X_train['Age_median'].var())

fig, ax = plt.subplots(figsize=(8,4))
X_train['Age'].plot(kind='kde', ax=ax, label='Original')
X_train['Age_mean'].plot(kind='kde', ax=ax, label='Mean')
X_train['Age_median'].plot(kind='kde', ax=ax, label='Median')
ax.legend(); ax.set_title('Age: Original vs Mean vs Median'); plt.show()

<h5 style="color:#78B89A; font-weight:bold;">7. When to use? → practical rule</h5>

**Mean:** useful when the numerical feature is reasonably symmetric and extreme values are not a major problem.

**Median:** useful when the feature is skewed or contains outliers.

Neither is automatically 'best'. Compare validation performance and distribution effects.

<h5 style="color:#78B89A; font-weight:bold;">8. Advantages / disadvantages</h5>

**Advantages:** simple, fast, keeps every row, easy to deploy.

**Disadvantages:** reduces variability, can create a spike around the imputed value, may weaken relationships between variables, and ignores other features.

<h5 style="color:#78B89A; font-weight:bold;">9. Modern scikit-learn way → SimpleImputer + Pipeline</h5>

Use `SimpleImputer` rather than manually calculating values when building ML systems. It learns the statistic on training data and reuses it on new data.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', LogisticRegression(max_iter=1000))
])

numeric_pipe.fit(X_train[['Age', 'Fare']], y_train)
# numeric_pipe.predict(X_test[['Age', 'Fare']])

<h5 style="color:#78B89A; font-weight:bold;">10. Final revision</h5>

```text
Numerical missing value
        ↓
Mean → symmetric / less affected by extremes
Median → skewed / outliers
        ↓
Learn from X_train
        ↓
Apply same statistic to X_test + production
```